In [ ]:
import os

import pandas as pd

from pynxtools_apm.examples.oasisb.batch_process import process_project
from pynxtools_apm.examples.oasisb.oasisb_utils import (
    get_project_id,
    snake_case_to_camel_case,
)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory: str = f"{fp.read().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory: str = f"{fp.read().strip().replace('/', os.sep)}"
print(trg_directory)

os.makedirs(f"{trg_directory.replace('/decompressed', '/pynxtools')}", exist_ok=True)
os.listdir(f"{trg_directory.replace('/decompressed', '')}")

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")

project_range: tuple[int, int] = (150, 156)
# TODO fix issues for project_id
# 36, some assignments broken
# 113 (too large allocations, likely unphysically large x, y, z coordinates)
# 149 some assignments broken
# bilal, (6, 6)
# pauly, (73, 73)
# uzuhashi (101, 101)

for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal in ("0", "1") and row.use in ("0", "1"):
        project_id = get_project_id(row.id)
        if project_range[0] <= int(project_id) <= project_range[1]:
            process_project(
                row.project_name,
                f"{src_directory}{os.sep}{row.project_name}.ods",
                f"{src_directory}{os.sep}aaa_legacy_data.bib",
                f"{src_directory}{os.sep}{row.project_name}.sha256.results.csv",
                trg_directory,
                f"{trg_directory.replace('/decompressed', '/pynxtools')}",
                openalex_file=f"{os.getcwd()}{os.sep}openalex/D_{snake_case_to_camel_case(row.project_name)}.json",
                # logger_file_path_suffix="test",
            )